# From notebook to application

This notebook opens the second block of Session 2. A notebook is useful for exploring data and explaining a result. Code that a teammate, a scheduled job or an application relies on needs a clear interface and a way to check that it still works. The transition is gradual: keep exploratory questions in the notebook and move repeated rules into functions, modules and, where useful, classes. The [workspace](../workspace/README.md) of this session contains both forms of the same listing analysis from the Session 1 case study: notebook cells in `notebooks/explore_listings.py` and the package `listingtools` with the class `ListingTable`. The [next lecture](03-lecture-git-github-and-ci.ipynb) shows how a team develops such a package together with Git, pull requests and continuous integration.

```mermaid
flowchart LR
  N["Notebook<br/>cells, global state"] --> F["Functions<br/>explicit inputs"]
  F --> M["Module<br/>importable .py file"]
  M --> P["Package and project<br/>src/, tests/, pyproject.toml"]
  P --> U["Used by notebooks,<br/>a command-line tool,<br/>later a service"]
  T["Tests"] -.->|check| P
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## What changes when code is reused?

A notebook kernel keeps names created by any cell that has run. Code can succeed even when its visible order is incomplete. A script starts a new Python process when launched, but it can still depend on files, environment variables or services outside the script. Reproducibility requires explicit inputs, a documented environment and checks—not merely changing the file extension from `.ipynb` to `.py`.

Compare a calculation tied to a global notebook variable with a function:

In [2]:
import pandas as pd

listings = pd.DataFrame({"room_type": ["Entire home/apt", "Private room", "Entire home/apt"]})
share = (listings["room_type"] == "Entire home/apt").mean()  # 2/3


def share_of(values: pd.Series, category: str) -> float:
    """Return the share matching category; return 0.0 for an empty Series."""
    if values.empty:
        return 0.0
    return float((values == category).mean())


print(share_of(listings["room_type"], "Entire home/apt"))

0.6666666666666666


The function names its inputs and defines an empty-input case. It can be imported and tested without first running a particular notebook cell. The result for missing values still needs a decision: by default, comparisons may treat them as non-matches, so the denominator is all rows. A different question might require dropping missing values first.

A **test** checks an expected result on controlled input. A useful first test covers an ordinary case and a boundary such as an empty Series. Tests do not prove every answer is correct; they make changes easier to review and catch known mistakes quickly.

## Scripts, modules and packages

A **script** is Python code run as a program, for example `python report.py`. A **module** is a `.py` file that can be imported, for example `import prices`. The same file can serve both roles. A **package** groups related modules under one import name, such as `listingtools.listings`. The course workspace uses a regular package with `__init__.py`; Python also supports namespace packages without that file.

Importing a module executes its top-level statements the first time it is imported in a process. For that reason, a reusable module should normally define functions and classes at the top level, and put program-only work inside `main()`:

In [3]:
# prices.py
def stay_cost(nightly_price: float, nights: int, fee: float) -> float:
    return nightly_price * nights + fee


def main() -> None:
    print(stay_cost(80, 3, 20))


if __name__ == "__main__":
    main()

260


`python prices.py` prints 260. `from prices import stay_cost` makes the function available without running `main()`. The import path depends on the interpreter's search path. Installing the course workspace package in its uv environment gives notebooks, tests and the command-line program the same importable code. Avoid naming a local file `pandas.py` or `json.py`: it may shadow the library you mean to import.

> 📖 **Read more:** [Session 1, extras/11-python-modules-and-packages.ipynb](../../01-careers-and-python/workbooks/extras/11-python-modules-and-packages.ipynb) (Whirlwind Tour of Python): the forms of the `import` statement (module, alias, selected names, and why `import *` is risky), a tour of the standard library, and installing third-party packages.

## A small project layout

The course workspace has this structure:

```text
workspace/
├── pyproject.toml
├── uv.lock
├── .github/workflows/ci.yml
├── notebooks/explore_listings.py
├── src/listingtools/
│   ├── __init__.py
│   ├── errors.py
│   ├── records.py
│   ├── listings.py
│   ├── text_features.py
│   ├── cli.py
│   └── __main__.py
├── tests/
└── solutions/
```

`pyproject.toml` describes the project and its dependencies; `uv.lock` records resolved versions. `src/listingtools/` holds code intended to be imported. `tests/` holds examples with expected outcomes. `notebooks/explore_listings.py` uses `# %%` cell markers so that VS Code can run it interactively. The `src` layout helps tests use the installed package instead of accidentally importing a same-named folder from the working directory. It is a choice, not a requirement for every one-file script.

From the workspace directory, `uv run pytest -q` runs the package tests in its project environment. From the repository root, `uv run --project sessions/02-software-engineering/workspace listing-summary` runs its command-line entry point. The [workspace README](../workspace/README.md) gives the data preparation command and the exercises; it is the operational guide for this example.

## Functions first, classes when useful

A function is often enough for a single calculation. A **class** (block 1) becomes useful when several operations share a coherent piece of state or when a named domain object makes the interface clearer. A small reminder of the pattern:

In [4]:
class Listing:
    def __init__(self, district: str, nightly_price: float) -> None:
        self.district = district
        self.nightly_price = nightly_price

    def stay_cost(self, nights: int, fee: float = 25) -> float:
        return self.nightly_price * nights + fee


flat = Listing("Pankow", 75)
print(flat.district)           # Pankow: attribute
print(flat.stay_cost(3))      # 250: method call

Pankow
250


When `flat.stay_cost(3)` runs, Python supplies `flat` as `self`. `__init__` initializes each instance. The workspace's `ListingTable` class wraps a DataFrame and offers operations such as loading, filtering and summarising listings. It is an example of one possible API, not a rule that every analysis needs a class.

A class can protect an invariant: for example, a `Listing` could reject a negative price or a non-positive number of nights. Such checks belong at the boundary where invalid input first enters the program. A method returning a new filtered table can also avoid surprising changes to the original table. These choices should be explicit in the API and covered by tests.

> 📖 **Read more:** [extras/01-classes-and-functions.ipynb](extras/01-classes-and-functions.ipynb) (Think Python): builds a `Time` class step by step: attributes, objects as return values, mutability and copying, pure functions, and two development plans (prototype and patch, design first), with exercises.
>
> 📖 **Read more:** [extras/02-classes-and-methods.ipynb](extras/02-classes-and-methods.ipynb) (Think Python): turns those functions into methods and adds static methods, `__init__`, `__str__`, comparison and operator overloading, with exercises.

## When to keep a notebook

A notebook remains valuable after logic moves into a module. It can state the question, show the data selection, call tested functions, draw a chart and interpret the result. A module holds transformations or calculations reused across notebooks, a command-line tool or a service. A scheduled notebook is also possible, but it still needs ordered execution, controlled inputs and checks. The relevant distinction is whether the result can be reproduced and maintained, not whether a file ends in `.ipynb`.

In the workspace, compare [the exploratory file](../workspace/notebooks/explore_listings.py) with [the package implementation](../workspace/src/listingtools/listings.py); the [exercise notebook](session-02-exercises.ipynb), part 2, starts from this comparison.

Canonical references: [Python modules](https://docs.python.org/3/tutorial/modules.html), [Python classes](https://docs.python.org/3/tutorial/classes.html), [Python Packaging User Guide: src layout](https://packaging.python.org/en/latest/discussions/src-layout-vs-flat-layout/) and [Good enough practices in scientific computing](https://doi.org/10.1371/journal.pcbi.1005510).

## Check your understanding

1. Name two ways in which a notebook result can depend on hidden state.
2. What is the difference between a script, a module and a package?
3. Why does program-only work go into `main()` under `if __name__ == "__main__":`?
4. What do `pyproject.toml`, `uv.lock`, `src/` and `tests/` each contain?
5. Give one reason to keep a notebook after the logic has moved into a package.

## Further reading

- Python Software Foundation (2026). *The Python Tutorial*, section 6 (modules). https://docs.python.org/3/tutorial/modules.html
- Python Packaging Authority (2026). *src layout vs flat layout*. https://packaging.python.org/en/latest/discussions/src-layout-vs-flat-layout/
- Astral (2026). *uv documentation: working on projects*. https://docs.astral.sh/uv/guides/projects/
- Wilson, G., Bryan, J., Cranston, K., Kitzes, J., Nederbragt, L., & Teal, T. K. (2017). Good enough practices in scientific computing. *PLOS Computational Biology*, 13(6), e1005510. https://doi.org/10.1371/journal.pcbi.1005510